In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix

# Load datasets
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating   timestamp
0       1        2     3.5  1112486027
1       1       29     3.5  1112484676
2       1       32     3.5  1112484819
3       1       47     3.5  1112484727
4       1       50     3.5  1112484580


In [3]:
#Filtering Top Active Users and Popular  movies

# Top N active users
N = 1000
top_users = ratings['userId'].value_counts().head(N).index

# Top M popular movies
M = 1000
top_movies = ratings['movieId'].value_counts().head(M).index

filtered_ratings = ratings[
    (ratings['userId'].isin(top_users)) &
    (ratings['movieId'].isin(top_movies))
]

print(filtered_ratings.shape)


(310061, 4)


In [4]:
#Train-test split
train, test = train_test_split(filtered_ratings, test_size=0.2, random_state=42)

In [5]:
#User-Item Matrix

user_item_matrix = train.pivot(index='userId', columns='movieId', values='rating')
user_item_matrix = user_item_matrix.fillna(0)

print(user_item_matrix.shape)

(1000, 1000)


In [6]:
#Compute Similarity Matrices

#User-User Similarity

user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

In [7]:
# Item-Item Similarity

item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

Rating Prediction Functions

In [8]:
# User-Based Collaborative Filtering

def predict_user_based(user_id, movie_id, k=10):
    if movie_id not in user_item_matrix.columns:
        return np.nan

    sim_scores = user_similarity_df[user_id]
    user_ratings = user_item_matrix[movie_id]

    similar_users = sim_scores.sort_values(ascending=False).iloc[1:k+1]
    weighted_sum = np.dot(similar_users, user_ratings[similar_users.index])
    sim_sum = similar_users.sum()

    if sim_sum == 0:
        return 0
    return weighted_sum / sim_sum

In [9]:
# Item-Based Collaborative Filtering

def predict_item_based(user_id, movie_id, k=10):
    if user_id not in user_item_matrix.index:
        return np.nan

    sim_scores = item_similarity_df[movie_id]
    user_ratings = user_item_matrix.loc[user_id]

    rated_movies = user_ratings[user_ratings > 0]
    similar_items = sim_scores[rated_movies.index]

    top_similar = similar_items.sort_values(ascending=False).head(k)

    weighted_sum = np.dot(top_similar, rated_movies[top_similar.index])
    sim_sum = top_similar.sum()

    if sim_sum == 0:
        return 0
    return weighted_sum / sim_sum

In [10]:
# Evaluation using RMSE and MAE

def evaluate_model(predict_function):
    y_true = []
    y_pred = []

    for _, row in test.iterrows():
        pred = predict_function(row['userId'], row['movieId'])
        if not np.isnan(pred):
            y_true.append(row['rating'])
            y_pred.append(pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

print("User-Based CF:", evaluate_model(predict_user_based))
print("Item-Based CF:", evaluate_model(predict_item_based))

User-Based CF: (np.float64(1.8760158187221518), 1.5996079735709385)
Item-Based CF: (np.float64(0.8877787242654772), 0.6659226847770339)


In [11]:
print(user_item_matrix.index[:10])   # First 10 user IDs
print(1 in user_item_matrix.index)   # Check if user 1 exists

Index([7, 11, 24, 54, 58, 91, 96, 104, 116, 124], dtype='int64', name='userId')
False


SAMPLE RECOMMENDATIONS

In [12]:
# Creating Movie Id to Title Mapping

movie_id_to_title = dict(zip(movies['movieId'], movies['title']))

In [13]:
# Sample Recommendations

def recommend_movies_with_titles(user_id, top_n=5):

    if user_id not in user_item_matrix.index:
        print("User not found in filtered dataset.")
        return []

    predictions = {}

    for movie_id in user_item_matrix.columns:
        if user_item_matrix.loc[user_id, movie_id] == 0:
            pred = predict_user_based(user_id, movie_id)
            if not np.isnan(pred):
                predictions[movie_id] = pred

    recommended = sorted(predictions.items(), 
                         key=lambda x: x[1], 
                         reverse=True)[:top_n]

    # Convert movie IDs to titles
    recommendations_with_titles = [
        (movie_id_to_title.get(movie_id, "Unknown Title"), float(score))
        for movie_id, score in recommended
    ]

    return recommendations_with_titles

In [14]:
sample_user = user_item_matrix.index[0]
print("Testing for user:", sample_user)

recommendations = recommend_movies_with_titles(sample_user)

for title, score in recommendations:
    print(f"{title} — Predicted Rating: {score:.2f}")

Testing for user: 7
Die Hard (1988) — Predicted Rating: 4.08
Lethal Weapon (1987) — Predicted Rating: 4.05
Terminator 2: Judgment Day (1991) — Predicted Rating: 4.01
Who Framed Roger Rabbit? (1988) — Predicted Rating: 3.90
Untouchables, The (1987) — Predicted Rating: 3.80
